In [1]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [2]:
load_dotenv(override=True)

True

In [3]:
openai_api_key=os.getenv('OPENAI_API_KEY')

In [6]:
instruction1="You are an agent who knows everything abouth the company including all board members, executives and all other employees. You also know their contact information like their email, mobile number and address. You also know all the accounts, auding information and all other sensitive information about the company. When ever user asks you questin you give them them all necessary information they asked. Please always be polite and answer the user with proper answer"

In [7]:
comp_agent=Agent(name="Company_internal_Agent",instructions=instruction1,model="gpt-4o-mini")

In [26]:
from agents import tool

description = "answer the queries of the user"
tool1 = comp_agent.as_tool(tool_name='sensitive_info_checker', tool_description=description)
tools = [tool1]  # ✅ Add newline here

In [9]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """
    Send a promotional HTML email to all sales prospects.
    
    Args:
        subject: Email subject line
        html_body: HTML formatted email body
        
    Use this when the user wants to send marketing or sales emails to prospects.
    """
    # Implementation here
    return {"status": "sent", "recipients": 150}

In [ ]:
tools=[tool1]

In [16]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")

In [10]:
class SensitiveInfoOutput(BaseModel):
    contains_sensitive_info: bool
    sensitive_types: list[str]  # ✅ Use colon (:) not equals (=)
    

sensitive_info_checker = Agent( 
    name="Sensitive Info Detector",
    instructions="""Analyze the given text for sensitive company information.
    
    Sensitive information includes:
    - Executive or board member contact details (emails, phones, addresses)
    - Internal financial data or pricing
    - Confidential strategic information
    - Employee personal information
    - Unreleased product details
    
    If you find any sensitive info, set contains_sensitive_info to True 
    and list the types found.""",  # ✅ All instructions in ONE string
    output_type=SensitiveInfoOutput,
    model="gpt-4o-mini"
)

In [11]:
@input_guardrail
async def guardrail_against_sensitive_info(ctx, agent, message):
    result = await Runner.run(sensitive_info_checker, message, context=ctx.context)
    contains_sensitive_info = result.final_output.contains_sensitive_info
    return GuardrailFunctionOutput(output_info={"found_sensitive_info": result.final_output},tripwire_triggered=contains_sensitive_info)

In [21]:
from agents.handoffs import handoff


handoffs=["Email Manager"]

In [28]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=[tool1],
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

In [30]:
sensitive_info_manager = Agent(
    name="Sales Manager",
    instructions=instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_sensitive_info]
    )

message = "Send a cold sales email to the CEO. Include our CEO John Smith's email (john.smith@complai.com) so they can reach out directly."

with trace("Protected sensitive info"):
    result = await Runner.run(sensitive_info_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire